In [ ]:
# import necessary libraries

import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import h5netcdf
import cartopy.crs as ccrs
import cartopy.feature as cfeature


In [ ]:
# read in necessary datasets: landfraction, temperature (tas), precipitation (pr), and area for any individual model
file_path_landfrac = r"insert landfraction path here"
landfrac_dataset = xr.open_dataset(file_path_landfrac)
file_path_temp = r"insert temperature path here"
temperature_dataset = xr.open_dataset(file_path_temp)
file_path_pr = r"insert precipitation path here"
pr_dataset = xr.open_dataset(file_path_pr)
file_path_area = r"insert area path here"
area_dataset = xr.open_dataset(file_path_area)

** landfrac values: 50 for CESM2, 30 for IPSL, 40 for UKESM, 35 for MPI LR, 45 for CNRM, 50 for MPI HR

In [ ]:
# interpolate the land‐fraction field onto the pr grid
mask_on_pr_grid = (
    landfrac_dataset["sftlf"]
      .interp(lat=pr_dataset.lat, lon=pr_dataset.lon)
      > np.nan # replace with model lanfrac value
)
pr_dataset_masked = pr_dataset.where(mask_on_pr_grid)

mask_on_tas_grid = (
    landfrac_dataset["sftlf"]
      .interp(lat=temperature_dataset.lat, lon=temperature_dataset.lon)
      > np.nan # replace with model landfrac value
)
temperature_dataset_masked = temperature_dataset.where(mask_on_tas_grid)

In [ ]:
# select necessary tas data for the specified time period
start = '2085-01-01'
end = '2099-12-30'

tas = temperature_dataset_masked['tas'].sel(time=slice(start, end))

In [ ]:
# select necessary pr data for the specified time period
pr = pr_dataset_masked['pr'].sel(time=slice(start, end))
pr_2 = (pr*60*60*24*30) # convert to mm/month
pr_converted = (pr*60*60*24*365.25) # convert to mm/year

In [ ]:
# specifiy lat and lon values
lat = pr['lat']
lon = pr['lon']

In [ ]:
# get mean pr per month
pr_monthly = pr_2.groupby('time.month').mean()
P_ann = pr_monthly.sum(dim='month') # get annual pr amount
P_ann.plot(vmin=0, vmax=2000) # sanity check

In [ ]:
MAT = tas.mean(dim='time') # get mean annual temp
MAP = pr_converted.mean(dim='time') # get mean annual pr

In [ ]:
# create seasonal masks: winter
winter_mask_NH = np.logical_or(pr.time.dt.month >= 10, pr.time.dt.month <= 3)  # Oct–Mar
winter_mask_SH = np.logical_and(pr.time.dt.month >= 4, pr.time.dt.month <= 9)  # Apr–Sep
#defining hemispheres by their position relative to the equator
is_NH = pr.lat >= 0
is_SH = pr.lat < 0

In [ ]:
# use these variables to "count months", they have a time dim = 120
# creating masks for each hemisphere - precip
pr_winter = xr.where(is_NH, pr.where(winter_mask_NH, pr), pr.where(winter_mask_SH, pr))
pr_summer = xr.where(is_NH, pr.where(~winter_mask_NH, pr), pr.where(~winter_mask_SH, pr))
#creating masks for each hemisphere - temp
tas_winter = xr.where(is_NH, tas.where(winter_mask_NH, tas), tas.where(winter_mask_SH, tas))
tas_summer = xr.where(is_NH, tas.where(~winter_mask_NH, tas), tas.where(~winter_mask_SH, tas))

In [ ]:
# get seasonal means for both precip and temp
tas_winter_mean = tas_winter.mean(dim='time')
tas_summer_mean = tas_summer.mean(dim='time')
pr_winter_mean = pr_winter.mean(dim='time')
pr_summer_mean = pr_summer.mean(dim='time')

In [ ]:
# DO NOT use these variables to count months, use them for broad criteria that doesn't depend on monthly data
# defining winter precip
P_winter = xr.where((tas_winter_mean > tas_summer_mean), pr_summer_mean, pr_winter_mean)
# defining summer precip
P_summer = xr.where((tas_winter_mean > tas_summer_mean), pr_winter_mean, pr_summer_mean)

In [ ]:
months = np.arange(1, 13) #1 = Jan, 12 = Dec
high_sun_NH = np.logical_and(months>=4, months<=9) # april-september
high_sun_SH = np.logical_or(months>=10, months<=3) # october-march

In [ ]:
# define hemispheres
pr_NH = pr_monthly.where(is_NH)
pr_SH = pr_monthly.where(is_SH)
# getting high sun months
pr_NH_var = pr_NH
pr_NH_high_sun = pr_NH_var.sel(month=high_sun_NH)
pr_SH_var = pr_SH
pr_SH_high_sun = pr_SH_var.sel(month=high_sun_SH)
# getting percentages of high sun month precip compared to total annual precip
pr_NH_high_sun_summed = pr_NH_high_sun.sum(dim=('month'))
pr_SH_high_sun_summed = pr_SH_high_sun.sum(dim=('month'))
P_NH = (pr_NH_high_sun_summed / P_ann) * 100
P_SH = (pr_SH_high_sun_summed / P_ann) * 100
# combining hemispheres to get entire globe
P = P_SH + P_NH
P.plot() # sanity check

In [ ]:
MAT_C = MAT - 273.15 # convert MAT to celsius (the aridity threshold formula uses celsius)

In [ ]:
# define and calculate aridity threshold using specified formula
aridity_thresh = ((MAT_C - 10) * 10) + (3 * P)
aridity_thresh.plot() # sanity check

In [ ]:
Tcold = temperature_dataset_masked['tas'].sel(time=slice(start, end)).groupby("time.month").mean().min(dim='month') # get avg coldest month
Thot = temperature_dataset_masked['tas'].sel(time=slice(start, end)).groupby("time.month").mean().max(dim='month') # get avg hottest month

ARID (B)
==

In [ ]:
B = xr.where((P_ann<(2*aridity_thresh)), True, False)
B.plot() # check

In [ ]:
B_area = (B*area_dataset['areacella']).sum()/area_dataset['areacella'].sum()*100

In [ ]:
BS = xr.where(np.logical_and(P_ann<2*aridity_thresh, P_ann>aridity_thresh), True, False)
BS.plot() # check

In [ ]:
BS_area = (BS*area_dataset['areacella']).sum()/area_dataset['areacella'].sum()*100

In [ ]:
BW_1 = xr.where(np.logical_and(P_ann<=aridity_thresh, B), True, False)
BW = xr.where(np.logical_and(BW_1, ~BS), True, False)
BW.plot() # check

In [ ]:
BW_area = (BW*area_dataset['areacella']).sum()/area_dataset['areacella'].sum()*100
(BW_area + BS_area) - B_area # sanity check, should = 0 (or a very small number)

In [ ]:
tas_1 = tas.sel(time=slice(start, end))
tas_2 = tas_1.groupby('time.month')
tas_monthly = tas_2.mean(dim='time')

crit = xr.where(tas_monthly > 283, True, False)
tas_crit = crit.sum(dim='month')

In [ ]:
BSh = xr.where(np.logical_and((tas_crit>8), BS), True, False)
BSh.plot() # check

In [ ]:
BSk = xr.where(np.logical_and(~BSh, BS), True, False)
BSk.plot() # check

In [ ]:
BWh = xr.where(np.logical_and((tas_crit>8), BW), True, False)
BWh.plot() # check

In [ ]:
BWk = xr.where(np.logical_and(~BWh, BW), True, False)
BWk.plot() # check

In [ ]:
# calculate land area percentages
BWh_area = (BWh*area_dataset['areacella']).sum()/area_dataset['areacella'].sum()*100
BWk_area = (BWk*area_dataset['areacella']).sum()/area_dataset['areacella'].sum()*100
BSh_area = (BSh*area_dataset['areacella']).sum()/area_dataset['areacella'].sum()*100
BSk_area = (BSk*area_dataset['areacella']).sum()/area_dataset['areacella'].sum()*100

In [ ]:
(BWh_area + BWk_area + BSh_area + BSk_area) - B_area # sanity check, should = 0 (or very small number)

TROPICAL (A)
==

In [ ]:
A_1 = xr.where((Tcold>(273.15+18)), True, False)
A = xr.where(np.logical_and(A_1, ~B), True, False)
A.plot() # check

In [ ]:
Pdry =  pr_converted.sel(time=slice(start, end)).groupby("time.month").mean().min(dim='month')

In [ ]:
Pmonth_1 = pr_converted.sel(time=slice(start, end))
Pmonth_2 = Pmonth_1.groupby('time.month')
Pmonth = Pmonth_2.mean(dim='time')

Ar_crit = xr.where(Pmonth>60, True, False)
Pmonth_crit = Ar_crit.sum(dim='month')

In [ ]:
rainforest_crit = Pmonth_crit > 10
Ar = xr.where(np.logical_and(rainforest_crit, A), True, False)
Ar.plot() # check

In [ ]:
Aw_crit = (pr_winter < 60)
Pwint_crit = Aw_crit.sum(dim='time')
Aw_month_crit = Pwint_crit > 2

Aw = xr.where(np.logical_and(A, ~Ar), True, False)
Aw.plot() # check

In [ ]:
# calculate land area percentages
Ar_area = (Ar*area_dataset['areacella']).sum()/area_dataset['areacella'].sum()*100
Aw_area = (Aw*area_dataset['areacella']).sum()/area_dataset['areacella'].sum()*100
A_area = (A*area_dataset['areacella']).sum()/area_dataset['areacella'].sum()*100

(Ar_area + Aw_area) - A_area # sanity check, should = 0 (or very small number)

SUBTROPICAL (C)
==

In [ ]:
tas_month_1 = tas.sel(time=slice(start, end))
tas_month_2 = tas_month_1.groupby('time.month')
tas_month = tas_month_2.mean(dim='time')

C_crit = xr.where(tas_month>283.15, True, False)
tas_month_crit = C_crit.sum(dim='month')

In [ ]:
month_count_crit = xr.where(np.logical_and(tas_month_crit >= 8, tas_month_crit <= 12), True, False)
C_unmasked = xr.where(month_count_crit, True, False)
not_tropic_arid = xr.where(np.logical_and(~B, ~A), True, False) #extract variables from arid and tropical so they can become arrays 
C = xr.where(np.logical_and(month_count_crit, not_tropic_arid), True, False)
C.plot() # check

In [ ]:
Cf = xr.where(np.logical_and(C, Pdry>30), True, False)
Cf.plot() # check

In [ ]:
Cm = xr.where(np.logical_and(C, ~Cf), True, False)
Cm.plot() # check

In [ ]:
C_area = (C*area_dataset['areacella']).sum()/area_dataset['areacella'].sum()*100

In [ ]:
Cm_area = (Cm*area_dataset['areacella']).sum()/area_dataset['areacella'].sum()*100
Cf_area = (Cf*area_dataset['areacella']).sum()/area_dataset['areacella'].sum()*100

(Cf_area + Cm_area) - C_area # sanity check, should = 0 (or very small number)

TEMPERATE (D)
==

In [ ]:
D_crit = xr.where(tas_month>283.15, True, False)
tas_month_crit = D_crit.sum(dim='month')

D_count_crit_1 = tas_month_crit >= 4
D_count_crit_2 = tas_month_crit <= 7
not_tropic_arid = xr.where(np.logical_and(~B, ~A), True, False)
D_1 = xr.where(np.logical_and(D_count_crit_1, D_count_crit_2), True, False)
D_2= xr.where(np.logical_and(D_1, not_tropic_arid), True, False)
D = xr.where(np.logical_and(D_2, ~C), True, False)
D.plot() # check

In [ ]:
DC = xr.where(np.logical_and(Tcold < 273.15, D), True, False)
DC.plot() # check

In [ ]:
DO = xr.where(np.logical_and(D, Tcold>273.15), True, False)
DO.plot() # check

In [ ]:
# calculate land area percentages
D_area = (D*area_dataset['areacella']).sum()/area_dataset['areacella'].sum()*100
DO_area = (DO*area_dataset['areacella']).sum()/area_dataset['areacella'].sum()*100
DC_area = (DC*area_dataset['areacella']).sum()/area_dataset['areacella'].sum()*100

D_area - (DO_area + DC_area) # sanity check, should = 0 (or very small number)

again, extremely small, so doesn't matter

SUBPOLAR (E)
==

In [ ]:
E_crit = xr.where(tas_month>283.15, True, False)
tas_month_crit = E_crit.sum(dim='month')

crit_1 = tas_month_crit >= 1
crit_2 = tas_month_crit <= 3
E_1 = xr.where(np.logical_and(crit_2, crit_1), True, False)
E_2 = xr.where(np.logical_and(not_tropic_arid, ~C), True, False)
E = xr.where(np.logical_and(E_1, E_2), True, False)
E.plot() # check

POLAR (F)
==

In [ ]:
F_1 = xr.where(np.logical_and(Thot < 283.15, ~E), True, False)
F_2 = xr.where(np.logical_and(F_1, not_tropic_arid), True, False)
F = xr.where(np.logical_and(F_2, ~C), True, False)
F.plot() # check

In [ ]:
crit_1 = Thot > 273.15
crit_2 = Thot < 283.15
Ft_1 = xr.where(np.logical_and(crit_1, crit_2), True, False)
Ft = xr.where(np.logical_and(Ft_1, F), True, False)
Ft.plot() # check

In [ ]:
monthly_temp = temperature_dataset_masked['tas'].groupby("time.month").mean(dim="time")
monthly_temp_2 = monthly_temp.mean('month')
Fi_1 = xr.where(monthly_temp_2 < 273.15, True, False)
Fi_2 = xr.where(np.logical_and(Fi_1, F), True, False)
Fi = xr.where(np.logical_and(Fi_2, ~Ft), True, False)
Fi.plot() # check

In [ ]:
# calculate land area percentages
F_area = (F*area_dataset['areacella']).sum()/area_dataset['areacella'].sum()*100
Fi_area = (Fi*area_dataset['areacella']).sum()/area_dataset['areacella'].sum()*100
Ft_area = (Ft*area_dataset['areacella']).sum()/area_dataset['areacella'].sum()*100

F_area - (Ft_area + Fi_area) # sanity check, should = 0 (or very small number)

NOVEL TROPICAL (xA)
==

(only exist in the future (2085-2099), do not use for historical data)

In [ ]:
xAr_crit = xr.where(tas_month>(273.15 + 40), True, False)
tas_month_crit = xAr_crit.sum(dim='month')

xAr_count_crit = tas_month_crit >= 1

xAr = xr.where(np.logical_and(Ar, xAr_count_crit), True, False)
xAr.plot()# check

In [ ]:
xAw_crit = xr.where(tas_month>(273.15 + 40), True, False)
tas_month_crit_2 = xAw_crit.sum(dim='month')

xAw_count_crit = tas_month_crit_2 >= 1

xAw_2 = xr.where(np.logical_and(Aw, xAw_count_crit), True, False)
xAw = xr.where(np.logical_and(xAw_2, tas_summer), True, False).mean(dim='time') # collapse time dim for 2D map
xAw.plot() # check

In [ ]:
xA = xAw + xAr # combine

xA.plot() # check

combine all these into a KTC dataset
==

In [ ]:
ds = xr.Dataset()
ds['lat'] = B.lat
ds['lon'] = B.lon

In [ ]:
ds['B'] = B
ds['B'].attrs= {'long_name' : 'arid'}

In [ ]:
ds['BS'] = BS
ds['BS'].attrs= {'long_name' : 'semi-arid/steppe'}
ds['BW'] = BW
ds['BW'].attrs= {'long_name' : 'desert'}

ds['BSh'] = BSh
ds['BSh'].attrs= {'long_name' : 'semi-arid/steppe hot'}
ds['BWh'] = BWh
ds['BWh'].attrs= {'long_name' : 'desert hot'}
ds['BSk'] = BSk
ds['BSk'].attrs= {'long_name' : 'semi-arid/steppe cold'}
ds['BWk'] = BWk
ds['BWk'].attrs= {'long_name' : 'desert cold'}

In [ ]:
ds['A'] = A
ds['A'].attrs= {'long_name' : 'tropical'}

ds['Ar'] = Ar
ds['Ar'].attrs= {'long_name' : 'rainforest'}
ds['Aw'] = Aw
ds['Aw'].attrs= {'long_name' : 'savannah'}

In [ ]:
ds['xA'] = xA
ds['xA'].attrs= {'long_name' : 'novel tropical'}

ds['xAr'] = xAr
ds['xAr'].attrs= {'long_name' : 'novel rainforest'}
ds['xAw'] = xAw
ds['xAw'].attrs= {'long_name' : 'novel savannah'}

In [ ]:
ds['C'] = C
ds['C'].attrs= {'long_name' : 'subtropical'}

ds['Cm'] = Cm
ds['Cm'].attrs= {'long_name' : 'w/ dry season'}
ds['Cf'] = Cf
ds['Cf'].attrs= {'long_name' : 'humid, w/o dry season'}

In [ ]:
ds['D'] = D
ds['D'].attrs={'long_name' : 'temperate'}

ds['DC'] = DC
ds['DC'].attrs={'long_name' : 'temperate continental'}
ds['DO'] = DO
ds['DO'].attrs={'long_name' : 'temperate oceanic'}

In [ ]:
ds['E'] = E
ds['E'].attrs={'long_name' : 'subpolar'}

In [ ]:
ds['F'] = F
ds['F'].attrs={'long_name' : 'polar'}

ds['Ft'] = Ft
ds['Ft'].attrs={'long_name' : 'polar tundra'}
ds['Fi'] = Fi
ds['Fi'].attrs={'long_name' : 'polar ice cap'}

In [ ]:
ds # should have 23 variables

In [ ]:
ds.to_netcdf(path='insert path here', mode='w') # save dataset